# BERT model za klasifikaciju emocija

## Učitavanje biblioteka i podešavanje okruženja

In [7]:
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    BertTokenizer,
    BertForSequenceClassification
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from data_reader import (
    get_training_data,
    get_validation_data,
    get_test_data
)

In [8]:
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## Učitavanje i priprema podataka

In [12]:
LABEL_TO_ID = {
    "anger": 0,
    "fear": 1,
    "joy": 2,
    "love": 3,
    "sadness": 4,
    "surprise": 5
}

ID_TO_LABEL = {
    label_id: label
    for label, label_id in LABEL_TO_ID.items()
}

NUM_LABELS = len(LABEL_TO_ID)

def encode_labels(data):
    data = data.copy()
    data["label"] = data["emotion"].map(LABEL_TO_ID)
    return data


train_data = encode_labels(get_training_data())
val_data = encode_labels(get_validation_data())
test_data = encode_labels(get_test_data())

print(f"Trening:    {len(train_data):>5} primera")
print(f"Validacija: {len(val_data):>5} primera")
print(f"Test:       {len(test_data):>5} primera")
print()
print("Klase:", LABEL_TO_ID)

train_data.head()


Trening:    15983 primera
Validacija:  1997 primera
Test:        2000 primera

Klase: {'anger': 0, 'fear': 1, 'joy': 2, 'love': 3, 'sadness': 4, 'surprise': 5}


,text,emotion,label
0,i didnt feel humiliated,sadness,4
1,i can go from feeling so hopeless to so damned...,sadness,4
2,im grabbing a minute to post i feel greedy wrong,anger,0
3,i am ever feeling nostalgic about the fireplac...,love,3
4,i am feeling grouchy,anger,0


## BERT tokenizer

Koristi se pretrained `bert-base-uncased` tokenizer koji tekst pretvara u numeričke tokene koje BERT model može da obradi.

In [13]:
MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

example_text = train_data.iloc[0]["text"]

tokens = tokenizer.tokenize(example_text)

print("Originalna rečenica:")
print(example_text)

print("\nTokeni:")
print(tokens)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Originalna rečenica:
i didnt feel humiliated

Tokeni:
['i', 'didn', '##t', 'feel', 'humiliated']


## PyTorch Dataset

Za svaki tekst tokenizer generiše `input_ids` i `attention_mask`, dok se odgovarajuća emocija prosleđuje kao numerička labela.

In [14]:
class EmotionDataset(Dataset):

    def __init__(self, data, tokenizer, max_length):
        self.texts = data["text"].tolist()
        self.labels = data["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

In [15]:
train_dataset = EmotionDataset(
    train_data,
    tokenizer,
    MAX_LENGTH
)

val_dataset = EmotionDataset(
    val_data,
    tokenizer,
    MAX_LENGTH
)

test_dataset = EmotionDataset(
    test_data,
    tokenizer,
    MAX_LENGTH
)

In [16]:
sample = train_dataset[0]

print("Input IDs shape:", sample["input_ids"].shape)
print("Attention mask shape:", sample["attention_mask"].shape)
print("Label:", sample["label"].item())
print("Emotion:", train_data.iloc[0]["emotion"])

Input IDs shape: torch.Size([128])
Attention mask shape: torch.Size([128])
Label: 4
Emotion: sadness


## DataLoader

DataLoader grupiše primere u batch-eve. Trening podaci se mešaju pri svakoj epohi, dok se validation i test podaci ne mešaju.

In [17]:
def create_data_loaders(batch_size):
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader, test_loader

In [18]:
train_loader, val_loader, test_loader = create_data_loaders(
    batch_size=16
)

batch = next(iter(train_loader))

print("Input IDs batch shape:", batch["input_ids"].shape)
print("Attention mask batch shape:", batch["attention_mask"].shape)
print("Labels batch shape:", batch["label"].shape)

Input IDs batch shape: torch.Size([16, 128])
Attention mask batch shape: torch.Size([16, 128])
Labels batch shape: torch.Size([16])


## BERT model

Koristi se pretrained `bert-base-uncased` model sa classification head-om prilagođenim za šest klasa emocija.

In [19]:
def create_model():
    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID
    )

    return model.to(device)

model = create_model()

print(model.classifier)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=6, bias=True)


## Provera prolaska jednog batch-a kroz model

In [20]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["label"].to(device)

with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss: 1.81720769405365
Logits shape: torch.Size([16, 6])


In [21]:
predictions = torch.argmax(
    outputs.logits,
    dim=1
)

first_prediction = predictions[0].item()
first_label = labels[0].item()

print("Predicted:", ID_TO_LABEL[first_prediction])
print("Actual:", ID_TO_LABEL[first_label])

Predicted: fear
Actual: joy


## Trening i validacija

Model se trenira na trening skupu, dok se nakon svake epohe evaluira na validation skupu.

Za poređenje konfiguracija prate se:
- Accuracy
- Macro F1

Najbolji checkpoint bira se prema najvećem validation Macro F1 rezultatu.

In [23]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()

    total_loss = 0
    all_predictions = []
    all_labels = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        predictions = torch.argmax(logits, dim=1)

        total_loss += loss.item()

        all_predictions.extend(
            predictions.detach().cpu().tolist()
        )

        all_labels.extend(
            labels.detach().cpu().tolist()
        )

    average_loss = total_loss / len(data_loader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return average_loss, accuracy, macro_f1

In [24]:
def validate(model, data_loader, device):
    model.eval()

    total_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            predictions = torch.argmax(
                logits,
                dim=1
            )

            total_loss += loss.item()

            all_predictions.extend(
                predictions.cpu().tolist()
            )

            all_labels.extend(
                labels.cpu().tolist()
            )

    average_loss = total_loss / len(data_loader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return average_loss, accuracy, macro_f1

## Trening jedne konfiguracije

In [26]:
def train_configuration(
    learning_rate,
    batch_size,
    epochs
):
    train_loader, val_loader, _ = create_data_loaders(
        batch_size
    )

    model = create_model()

    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate
    )

    best_validation_f1 = -1
    best_validation_accuracy = 0
    best_epoch = 0
    best_model_state = None

    history = []

    for epoch in range(epochs):
        train_loss, train_accuracy, train_macro_f1 = train_epoch(
            model,
            train_loader,
            optimizer,
            device
        )

        val_loss, val_accuracy, val_macro_f1 = validate(
            model,
            val_loader,
            device
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "train_macro_f1": train_macro_f1,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "val_macro_f1": val_macro_f1
        })

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"train loss: {train_loss:.4f}, "
            f"train acc: {train_accuracy:.4f}, "
            f"train F1: {train_macro_f1:.4f} | "
            f"val loss: {val_loss:.4f}, "
            f"val acc: {val_accuracy:.4f}, "
            f"val F1: {val_macro_f1:.4f}"
        )

        if val_macro_f1 > best_validation_f1:
            best_validation_f1 = val_macro_f1
            best_validation_accuracy = val_accuracy
            best_epoch = epoch + 1

            best_model_state = {
                key: value.cpu().clone()
                for key, value in model.state_dict().items()
            }

    return {
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "epochs": epochs,
        "best_epoch": best_epoch,
        "validation_accuracy": best_validation_accuracy,
        "validation_macro_f1": best_validation_f1,
        "history": history,
        "model_state": best_model_state
    }

## Podešavanje hiperparametara

Testira se više kombinacija learning rate-a i batch size-a.

Najbolja konfiguracija bira se prema najvećem validation Macro F1 rezultatu.

In [27]:
EXPERIMENTS = [
    {
        "learning_rate": 2e-5,
        "batch_size": 16,
        "epochs": 3
    },
    {
        "learning_rate": 3e-5,
        "batch_size": 16,
        "epochs": 3
    },
    {
        "learning_rate": 5e-5,
        "batch_size": 16,
        "epochs": 3
    },
    {
        "learning_rate": 2e-5,
        "batch_size": 32,
        "epochs": 3
    }
]

In [28]:
experiment_results = []

best_result = None

for experiment_index, experiment in enumerate(EXPERIMENTS, start=1):

    print("\n" + "=" * 70)

    print(
        f"Experiment {experiment_index}/{len(EXPERIMENTS)} | "
        f"LR={experiment['learning_rate']} | "
        f"Batch size={experiment['batch_size']} | "
        f"Epochs={experiment['epochs']}"
    )

    print("=" * 70)

    result = train_configuration(
        learning_rate=experiment["learning_rate"],
        batch_size=experiment["batch_size"],
        epochs=experiment["epochs"]
    )

    experiment_results.append(result)

    if (
        best_result is None
        or result["validation_macro_f1"]
        > best_result["validation_macro_f1"]
    ):
        best_result = result


Experiment 1/4 | LR=2e-05 | Batch size=16 | Epochs=3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | train loss: 0.4467, train acc: 0.8379, train F1: 0.7862 | val loss: 0.1596, val acc: 0.9324, val F1: 0.9038
Epoch 2/3 | train loss: 0.1314, train acc: 0.9416, train F1: 0.9113 | val loss: 0.1401, val acc: 0.9364, val F1: 0.9114
Epoch 3/3 | train loss: 0.0975, train acc: 0.9538, train F1: 0.9277 | val loss: 0.1259, val acc: 0.9384, val F1: 0.9134

Experiment 2/4 | LR=3e-05 | Batch size=16 | Epochs=3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | train loss: 0.3967, train acc: 0.8582, train F1: 0.8136 | val loss: 0.1607, val acc: 0.9319, val F1: 0.9076
Epoch 2/3 | train loss: 0.1398, train acc: 0.9381, train F1: 0.9064 | val loss: 0.1322, val acc: 0.9359, val F1: 0.9076
Epoch 3/3 | train loss: 0.1165, train acc: 0.9429, train F1: 0.9111 | val loss: 0.1279, val acc: 0.9364, val F1: 0.9106

Experiment 4/4 | LR=2e-05 | Batch size=32 | Epochs=3


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 | train loss: 0.6027, train acc: 0.7888, train F1: 0.7109 | val loss: 0.1944, val acc: 0.9269, val F1: 0.9053
Epoch 2/3 | train loss: 0.1476, train acc: 0.9381, train F1: 0.9068 | val loss: 0.1600, val acc: 0.9339, val F1: 0.9121
Epoch 3/3 | train loss: 0.1060, train acc: 0.9500, train F1: 0.9213 | val loss: 0.1291, val acc: 0.9379, val F1: 0.9106


## Rezultati podešavanja hiperparametara

In [29]:
tuning_results = pd.DataFrame([
    {
        "learning_rate": result["learning_rate"],
        "batch_size": result["batch_size"],
        "epochs": result["epochs"],
        "best_epoch": result["best_epoch"],
        "validation_accuracy": result["validation_accuracy"],
        "validation_macro_f1": result["validation_macro_f1"]
    }
    for result in experiment_results
])

tuning_results

,learning_rate,batch_size,epochs,best_epoch,validation_accuracy,validation_macro_f1
0,0.00002,16,3,3,0.938408,0.913409
1,0.00003,16,3,3,0.942914,0.918195
2,0.00005,16,3,3,0.936405,0.910596
3,0.00002,32,3,2,0.933901,0.912074


In [30]:
tuning_results = tuning_results.sort_values(
    by="validation_macro_f1",
    ascending=False
).reset_index(drop=True)

tuning_results

,learning_rate,batch_size,epochs,best_epoch,validation_accuracy,validation_macro_f1
0,0.00003,16,3,3,0.942914,0.918195
1,0.00002,16,3,3,0.938408,0.913409
2,0.00002,32,3,2,0.933901,0.912074
3,0.00005,16,3,3,0.936405,0.910596


In [31]:
print("Najbolja BERT konfiguracija:")
print(f"Learning rate: {best_result['learning_rate']}")
print(f"Batch size: {best_result['batch_size']}")
print(f"Best epoch: {best_result['best_epoch']}")
print(
    f"Validation accuracy: "
    f"{best_result['validation_accuracy']:.4f}"
)
print(
    f"Validation macro F1: "
    f"{best_result['validation_macro_f1']:.4f}"
)

Najbolja BERT konfiguracija:
Learning rate: 3e-05
Batch size: 16
Best epoch: 3
Validation accuracy: 0.9429
Validation macro F1: 0.9182


In [32]:
tuning_results.to_csv(
    "results/bert_tuning_results.csv",
    index=False
)